# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We follow FAIR and MLCroissant best practices, referencing all data entities (record sets, fields/columns) by their `@id`.

### Dataset Source
The dataset Croissant schema is available at: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset's metadata and examine the top-level information.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access basic metadata
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}\n")
print(f"Keywords: {getattr(metadata, 'keywords', None)}\n")

## 2. Data Overview
Let's review the available record sets and fields/columns that can be loaded for analysis.

- Each record set, field, and column is referenced by its `@id` per Croissant.
- We'll extract a list of record sets and their fields, and display their IDs and labels.


In [ ]:
# List all available record sets and their fields by @id
record_sets = dataset.metadata.recordSet
print("Available Record Sets and Fields:")
record_set_ids = []
record_set_fields = {}

for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']} | name: {rs.get('name')}")
    record_set_ids.append(rs['@id'])
    fields = rs.get('field', [])
    field_ids = []
    for field in fields:
        # Field is @id or dict
        if isinstance(field, dict):
            fid = field.get('@id')
            label = field.get('name', field.get('label', ''))
            dtype = field.get('dataType', '')
        else:
            fid = field
            label = ''
            dtype = ''
        print(f"   - Field @id: {fid} | {label} | {dtype}")
        field_ids.append(fid)
    record_set_fields[rs['@id']] = field_ids
if not record_set_ids:
    print("No record sets were found. Please refer to the Croissant schema for details.")

## 3. Data Extraction
We'll extract data for one or more record sets, by their `@id`, into pandas DataFrames for further analysis.

If there is no record set found, please inspect the Croissant schema's `distribution` and field organization to load records.


In [ ]:
# Example: load data for each record set directly by @id
dataframes = {}

if record_set_ids:
    for record_set_id in record_set_ids:
        print(f"Loading records for record set {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        display(df.head())
else:
    print("No record sets found. Trying to load data from main distribution as fallback.")
    # As a fallback, try loading records from main distribution
    try:
        records = list(dataset.records())
        df = pd.DataFrame(records)
        dataframes['main'] = df
        print("Columns:", df.columns.tolist())
        display(df.head())
    except Exception as e:
        print("No records available for loading. Please check the Croissant schema structure.")

## 4. Exploratory Data Analysis (EDA)
Let's perform EDA steps, such as filtering records, normalizing numeric columns, and grouping by a categorical variable.

Adjust `numeric_field_id` and `group_field_id` as appropriate for your target analysis—the exact field `@id`s will correspond to those listed in the record set in step 2.


In [ ]:
# Select the primary record set and numeric field for EDA

import numpy as np

# Choose a record set:
if dataframes:
    main_record_set_id = next(iter(dataframes.keys()))  # pick the first if multiple
    df = dataframes[main_record_set_id]
    print(f"Using record set: {main_record_set_id}")

    # Try to infer a numeric field:
    # Let's try 'cr:peptide_count' if available, otherwise pick the first float/integer column
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        if 'peptide' in col.lower() or 'intensity' in col.lower() or 'quantity' in col.lower() or 'abundance' in col.lower():
            numeric_field_id = col
        if 'sample' in col.lower() or 'condition' in col.lower():
            group_field_id = col
    if not numeric_field_id:
        # fallback: pick first numeric column
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    if not group_field_id:
        # fallback: pick first non-numeric column that isn't an ID
        for col in df.columns:
            if pd.api.types.is_string_dtype(df[col]) and not col.lower().endswith('id'):
                group_field_id = col
                break

    if numeric_field_id:
        print(f"Numeric field inferred: {numeric_field_id}")
        threshold = np.nanquantile(df[numeric_field_id], 0.7) if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold} (top 5):")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} (top 5):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped mean of numeric columns by {group_field_id} (top 5):")
            display(grouped_df.head())
        else:
            print("(No groupable categorical field inferred. Skipping grouping.)")
    else:
        print("No numeric field suitable for EDA detected.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions and relationships. Adjust the field IDs for your context.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if 'filtered_df' in locals() and not filtered_df.empty and numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(filtered_df[numeric_field_id], bins=25, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No data available for plotting.")

## 6. Conclusion
In this notebook we demonstrated how to explore a Croissant-described dataset using `mlcroissant`. We:

- Loaded metadata and data records by referencing all entities by their `@id` as per the FAIR and Croissant standards.
- Provided an overview of record sets and their available fields.
- Loaded data into DataFrames and performed exploratory filtering, normalization, grouping, and initial visualizations.

To go further, consider domain-specific analyses or model building using this mass spectrometry protein dataset.
